In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/lra-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 04 · The same patterns with ADK 2 `Workflow` (optional: the `adk` extra)

Google's ADK 2 gives you the primitives natively: **interrupts** (`RequestInput`), **resumability** (`ResumabilityConfig`), **re-run or complete on resume** (`rerun_on_resume`, for the node that interrupted), **routing** (`ctx.route`) and pluggable **session stores** (SQLite → Cloud SQL). This notebook runs the ticket-queue graph in `examples/adk_ticket_queue/nightly_workflow.py` *without a model* so you can see the mechanics; `use_model=True` puts Gemini in the `plan` node.

Needs `pip install -e ".[adk]"` (ADK 2 and the Google Cloud clients, about 220 MB; measured 2026-09-26, verify). Without it, `scripts/run_notebooks.py` skips this notebook; every other notebook runs on the default install.

```mermaid
flowchart LR
  S((START)) --> B[agree_budget<br/>RequestInput 'budget'<br/>rerun_on_resume=True]
  B --> P[plan<br/>judgement]
  P --> Q[queue_up<br/>side effect, runs once]
  Q --> C[check_front<br/>RequestInput 'wake'<br/>rerun_on_resume=True]
  C -- ready --> Y[buy<br/>idempotency key]
  C -- sold_out --> A[abandon]
```
(Cells use top-level `await`: Jupyter already runs an event loop.)

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..").resolve() / "examples" / "adk_ticket_queue"))   # from notebooks/ or solutions/

from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from nightly_workflow import ASK_BUDGET, WAKE, Venue, build_app, check_adk_workflow, run_until_interrupt

venue = Venue()
svc = InMemorySessionService()          # production: DatabaseSessionService("postgresql+asyncpg://...") or VertexAiSessionService
runner = Runner(app=build_app(venue), session_service=svc)
session = await svc.create_session(app_name="nightly_app", user_id="u1",
                                   state={"events": [{"id": "ams-tue", "weekday": "Tuesday"}, {"id": "ams-sat", "weekday": "Saturday"}]})
async def state(s=svc, sid=None):
    return (await s.get_session(app_name="nightly_app", user_id="u1", session_id=sid or session.id)).state

r1 = await run_until_interrupt(runner, "u1", session.id, text="Get us two tickets")
print("stopped on:", r1["interrupts"], "| venue calls:", venue.calls)
assert r1["interrupts"] == [ASK_BUDGET]

The graph asked a question and **stopped**. The invocation lives in the session store; the Python process could exit here.

Answer it by resuming the *same invocation* with a `FunctionResponse` whose id is the interrupt id:

In [ ]:
r2 = await run_until_interrupt(runner, "u1", session.id, invocation_id=r1["invocation_id"], answers={ASK_BUDGET: {"budget": 250}})
print("stopped on:", r2["interrupts"], "| venue calls:", venue.calls, "| state:", {k: v for k, v in (await state()).items() if k != "events"})
assert r2["interrupts"] == [WAKE] and venue.calls == ["join_queue"]

Four nodes ran off one answer: `plan` chose Saturday (the weekend rule), `queue_up` took **one** ticket, `check_front` found us at #14,203 and parked on `wake`.

## Wake-ups (what Cloud Scheduler → Pub/Sub → `/wake` does)

In [ ]:
ticket = (await state())["ticket"]
venue.advance(ticket, 10_000)                        # the queue moves while nothing of ours runs
r3 = await run_until_interrupt(runner, "u1", session.id, invocation_id=r2["invocation_id"], answers={WAKE: {"ok": True}})
print("wake 1 -> stopped on:", r3["interrupts"], "| join_queue calls:", venue.calls.count("join_queue"), "| position:", venue.position(ticket))
venue.advance(ticket, 10_000)
r4 = await run_until_interrupt(runner, "u1", session.id, invocation_id=r3["invocation_id"], answers={WAKE: {"ok": True}})
print("wake 2 -> stopped on:", r4["interrupts"], "| venue calls:", venue.calls)
print("order:", (await state())["order"])
assert venue.calls == ["join_queue", "purchase"] and r4["interrupts"] == []

`queue_up` never re-ran across the wake-ups: a resume continues from the node that interrupted, and nodes that already finished are not replayed. `check_front`, the node that interrupted, re-ran on every wake-up because it is marked `rerun_on_resume=True`; with `False` ADK would mark it complete and take the resume input (`{"ok": true}`) as its output, without looking at the queue. `buy` executed once, with an idempotency key.

## Staleness guard: the world changed while we waited

In [ ]:
venue2 = Venue(); svc2 = InMemorySessionService(); runner2 = Runner(app=build_app(venue2), session_service=svc2)
s2 = await svc2.create_session(app_name="nightly_app", user_id="u1", state={"events": [{"id": "ams-sat", "weekday": "Saturday"}]})
a = await run_until_interrupt(runner2, "u1", s2.id, text="go")
b = await run_until_interrupt(runner2, "u1", s2.id, invocation_id=a["invocation_id"], answers={ASK_BUDGET: {"budget": 200}})
t2 = (await state(svc2, s2.id))["ticket"]
venue2.advance(t2, 99_999); venue2.sell_out("ams-sat")
await run_until_interrupt(runner2, "u1", s2.id, invocation_id=b["invocation_id"], answers={WAKE: {"ok": True}})
print("calls:", venue2.calls, "| orders:", venue2.orders, "| routed to abandon:", (await state(svc2, s2.id))["order"] is None)
assert venue2.orders == {} and "purchase" not in venue2.calls

## The classic mistake: a new invocation instead of a resume

Typing in the chat box (no `invocation_id`) starts a **new** invocation: the graph replays from START, and answering the budget again takes a second queue ticket.

In [ ]:
venue3 = Venue(); svc3 = InMemorySessionService(); runner3 = Runner(app=build_app(venue3), session_service=svc3)
s3 = await svc3.create_session(app_name="nightly_app", user_id="u1", state={"events": [{"id": "ams-sat", "weekday": "Saturday"}]})
a = await run_until_interrupt(runner3, "u1", s3.id, text="go")
await run_until_interrupt(runner3, "u1", s3.id, invocation_id=a["invocation_id"], answers={ASK_BUDGET: {"budget": 200}})
c = await run_until_interrupt(runner3, "u1", s3.id, text="where am I in line?")       # WRONG: a new invocation
await run_until_interrupt(runner3, "u1", s3.id, invocation_id=c["invocation_id"], answers={ASK_BUDGET: {"budget": 200}})
print("join_queue calls:", venue3.calls.count("join_queue"), "<- two tickets")
assert venue3.calls.count("join_queue") == 2

## Your turn: build the graph

Complete the nodes; `check_adk_workflow` runs your graph through a budget answer and three wake-ups:

* `agree_budget` interrupts with `interrupt_id=ASK_BUDGET` until the budget arrives, and **re-runs on resume**.
* `queue_up` is a side effect: one ticket across all wake-ups (it finishes before any interrupt, so resuming the same invocation never replays it).
* `check_front` interrupts while we are still queued and must re-check the world on every wake-up (`rerun_on_resume`?); at the front it routes to `"ready"` with at least two seats left, else `"sold_out"`.
* Wire `START → agree_budget → plan → queue_up → check_front → {"ready": buy, "sold_out": abandon}`.

In [ ]:
# TODO: fill in the blanks: ask_budget, check_rerun, route, edges
# YOUR CODE HERE: replace each ____ in this cell, then delete the next line
raise NotImplementedError("fill in the blanks: ask_budget, check_rerun, route, edges")
from google.adk.events.request_input import RequestInput
from google.adk.workflow import START, Workflow, node

def build_workflow(venue):
    @node(rerun_on_resume=True)
    def agree_budget(ctx):
        said = ((ctx.resume_inputs or {}).get(ASK_BUDGET) or {}).get("budget")
        if not said:
            return ____
        ctx.state["budget_per_seat"] = float(said)
        return {"budget_per_seat": float(said)}

    @node
    def plan(ctx):
        ctx.state["event_id"] = ctx.state["events"][0]["id"]
        return {"event_id": ctx.state["event_id"]}

    @node                                            # finished before any interrupt: a resume never replays it
    def queue_up(ctx):
        t = venue.join_queue(ctx.state["event_id"], idempotency_key=f"{ctx.session.id}:{ctx.state['event_id']}")
        ctx.state["ticket"] = t["ticket"]
        return t

    @node(rerun_on_resume=____)           # re-check the world, or take the wake-up payload as the answer?
    def check_front(ctx):
        pos = venue.position(ctx.state["ticket"])
        if pos > 0:
            return RequestInput(interrupt_id=WAKE, message=f"Still at #{pos}")
        left = venue.seats_left(ctx.state["event_id"])     # staleness guard: re-verify before acting
        ctx.route = ____
        return {"position": pos, "seats_left": left}

    @node
    def buy(ctx):
        order = venue.purchase(ctx.state["event_id"], 2, idempotency_key=f"{ctx.session.id}:{ctx.state['event_id']}:purchase")
        ctx.state["order"] = order
        return order

    @node
    def abandon(ctx):
        ctx.state["order"] = None
        return {"abandoned": True}

    return Workflow(name="practice", edges=[____])

print(check_adk_workflow(build_workflow))

## Deploying this shape on Google Cloud

| Local | Cloud | Change |
|---|---|---|
| `InMemorySessionService` | Cloud SQL (Postgres) via `DatabaseSessionService("postgresql+asyncpg://...?host=/cloudsql/...")`, or Agent Runtime Sessions | a connection string |
| `adk web` | one Cloud Run service (`examples/adk_ticket_queue/main.py`) | a Dockerfile and an entry point |
| you calling `run_until_interrupt` | Cloud Scheduler → Pub/Sub → push → `POST /wake` | `trigger_sources=["pubsub"]` plus the custom `/wake` |
| `Venue` | the real API with an `Idempotency-Key` header | none |

ADK's built-in Pub/Sub trigger route **creates a new session per message**; a long-running run needs a wake endpoint that resumes an *existing* session, which is what `main.py` adds. The design questions for this notebook are in the topic primer's drills (`../../PRIMER.md` §11, part D).

## Takeaways

* `RequestInput` is one primitive for both "waiting for a person" and "waiting for the world".
* `rerun_on_resume` decides what a resumed node does: re-run and re-check (`True`) or take the resume input as its answer (`False`). Side effects stay safe because finished nodes are not replayed and they carry idempotency keys.
* Resume the invocation; don't start a new one.
* Prompts can't wake themselves; something with a clock has to call the endpoint.